In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
# CELL 2: Load the model and tokenizer
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Can read up to 2048 tokens in one go
dtype = None          # Auto-detects your GPU type
load_in_4bit = True   # Shrink memory size by 4x to fit on the free tier!

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Set up LoRA adapters - we only train ~1% of the model's weights!
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
print("Model loaded successfully into ultra-low memory mode!")

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:153: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2026.6.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Model loaded successfully into ultra-low memory mode!


In [ ]:
# CELL 3: Load and format your data.json file
import json
from datasets import Dataset

# Define the Alpaca format layout
alpaca_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

# Load the file you uploaded
with open("data.json", "r") as f:
    custom_data = json.load(f)

# Format helper to structure data into the prompt format
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = alpaca_prompt.format(instruction, output) + "<|end_of_text|>"
        texts.append(text)
    return { "text" : texts }

# Convert JSON data into a Dataset object
dataset = Dataset.from_list(custom_data)
dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"Successfully loaded and formatted {len(dataset)} QA pairs from data.json!")

Map:   0%|          | 0/96 [00:00<?, ? examples/s]

Successfully loaded and formatted 96 QA pairs from data.json!


In [ ]:
# CELL 4: Run the actual fine-tuning
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        # num_train_epochs = 1, # For longer training runs!
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

trainer_stats = trainer.train()
print("🎉 All done! Your Vellium-expert model is now fine-tuned.")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/96 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 96 | Num Epochs = 5 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,3.604300
2,3.719100
3,3.504500
4,3.337300
5,3.313400
6,2.954200
7,2.800700
8,2.361800
9,2.196300
10,1.986200


🎉 All done! Your Vellium-expert model is now fine-tuned.


In [ ]:
# CELL 5: Test your newly trained model
# @title Show final memory and time stats


FastLanguageModel.for_inference(model)

inputs = tokenizer(
[
    alpaca_prompt.format(
        "What tech stack does Vellium use?", # Test query from your dataset
        ""
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
print(tokenizer.decode(outputs[0]))

<|begin_of_text|>Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What tech stack does Vellium use?

### Response:
Frontend: React 18, TypeScript, Vite, Tailwind CSS. Backend: Express.js on Node.js. AI: @google/genai SDK for Gemini API calls.<|end_of_text|>


In [ ]:
# CELL 6: Log in to your Hugging Face account
from huggingface_hub import login
login()

In [ ]:
# CELL 7: Save, quantize to GGUF, and upload directly to your profile
model.push_to_hub_gguf(
    "Kathit/vellium-llama3-8b-gguf",
    tokenizer,
    quantization_method = "q4_k_m", # Excellent balance of small size and high logic quality
    token = True
)
print("Conversion complete! Your standalone GGUF file is live at huggingface.co/Kathit/vellium-llama3-8b-gguf")

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/896 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Unsloth: Merge process complete. Saved to `/tmp/unsloth_gguf_wc029wvj`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b9789-mix-1f1aaa4 (app-b9789-mix-1f1aaa4-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/tmp/unsloth_gguf_wc029wvj_gguf/llama-3.1-8b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversio

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...1-8b-instruct.Q4_K_M.gguf:   0%|          |  559kB / 4.92GB            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/Kathit/vellium-llama3-8b-gguf
Unsloth: Cleaning up temporary files...
Conversion complete! Your standalone GGUF file is live at huggingface.co/Kathit/vellium-llama3-8b-gguf


Step-by-Step Native Terminal Pull
Go to your Hugging Face model page: https://huggingface.co/Kathit/vellium-llama3-8b-gguf.

Click the white Use this model button in the top right corner  choose Ollama.

Copy the string it gives you. It looks like this:

ollama run hf.co/Kathit/vellium-llama3-8b-gguf:Q4_K_M

Now, open your computer's regular terminal application:

Windows: Open PowerShell or Command Prompt (CMD).

Linux/WSL: Open your standard terminal shell.

Paste the string you copied, but change the word run to pull. Type this command and press Enter:

ollama pull hf.co/Kathit/vellium-llama3-8b-gguf:Q4_K_M

You will see a clean download progress bar right inside your terminal as Ollama directly fetches the compressed weights from Hugging Face.

Once the native terminal pull reaches 100%, the model is automatically activated inside your system! You do not need to move any files or change directories.

Because you used the ollama run command, it will automatically open an active chat prompt right inside that terminal window as soon as the download finishes.

Here is exactly what to do next depending on how you want to chat with it:

Option A: Chat Directly in Your Terminal (Right Now)
Wait for the download progress bars to completely disappear.

You will see a prompt line show up that looks like this: >>> Send a message (/? for help)

Type any question from your data file directly into your terminal and hit Enter.

To exit the terminal chat when you are done, simply type /bye and hit Enter.

Run the Activation Command
Type this exact command and press Enter to activate again:

ollama run hf.co/Kathit/vellium-llama3-8b-gguf:Q4_K_M



Option B: Activate It in Your Open WebUI Interface

Because Ollama handles the model management globally behind the scenes, pulling it via your native terminal has completely skipped the UI restrictions.

Open your web browser tab back to your Open WebUI page (http://localhost:3000).

Simply refresh the page (or press F5 / Ctrl+R) to force the UI to check your local model library database.

Click on the Model Selector dropdown block at the bottom right of your chat screen.

Look through your available choices. You will see Kathit/vellium-llama3-8b-gguf:Q4_K_M sitting cleanly inside the list!

Click it, type a question, and start chatting through the graphic interface.



Pro-Tip: Create a Short Nickname (Optional)
If typing that massive Hugging Face link every time is annoying, you can give your model a short 1-word nickname so it's super easy to launch.

Open your terminal and run this copy command once:


ollama cp hf.co/Kathit/vellium-llama3-8b-gguf:Q4_K_M vellium

Now, anytime you open your terminal in the future, you can launch your fine-tuned model by simply typing:


ollama run vellium

Whenever you are done chatting, just type /bye like before to close it out!


Now do,

ollama cp vellium kathitjoshi/vellium:latest


ollama push kathitjoshi/vellium:latest

to make your model available to others

Here is the complete, hallucination-free, and straightforward loop for whenever you want to update your data. It breaks down into three simple phases: updating in the cloud, clearing out the old files locally, and pulling down the fresh update.

 Phase 1: Update & Push on Google Colab
Open your Google Colab notebook and make sure your T4 GPU runtime is connected.

Click the Folder Icon on the left sidebar and upload your newly updated data.json file to overwrite the old one.

Skip Cells 1 and 2 entirely! Start running from Cell 3:

Run Cell 3 (The Loader): To parse the new question-and-answer dataset.

Run Cell 4 (The Trainer): To fine-tune the model's weights on your new updates.

Run Cell 6: To verify your Hugging Face login state.

Run your custom storage-saving Cell 7:
(This merges, converts to GGUF, deletes the massive temporary 16-bit clutter, and uploads the final lightweight .gguf file to your Hugging Face repo automatically).

Python
# YOUR CUSTOM CELL 7:
model.save_pretrained_merged("merged_16bit_model", tokenizer, save_method = "merged_16bit")
model.save_pretrained_gguf("gguf_output", tokenizer, quantization_method = "q4_k_m")

import shutil
shutil.rmtree("merged_16bit_model") # Prevents "Disk Full" errors

from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path="gguf_output",
    repo_id="Kathit/vellium-llama3-8b-gguf",
    repo_type="model"
)
Click Runtime  Disconnect and delete runtime at the top menu to turn off Colab and stop burning your cloud usage time.

 Phase 2: Clear the Cache on Your Local Laptop
Because you are using the exact same model name (Kathit/vellium-llama3-8b-gguf:Q4_K_M), your laptop's local Ollama instance thinks it already has the file and will ignore the update unless you force it to look again.

Open your computer's native terminal (PowerShell on Windows or Terminal on Linux/WSL) and run this command to erase the stale cache:


ollama rm hf.co/Kathit/vellium-llama3-8b-gguf:Q4_K_M
ollam rm vellium

 Phase 3: Pull & Chat with Your Updated Model

Now, run the activation command directly in that same terminal window:


ollama run hf.co/Kathit/vellium-llama3-8b-gguf:Q4_K_M

What happens: Ollama sees that the model is missing locally, connects to Hugging Face, fetches your brand-new updated q4_k_m GGUF file, and activates the live chat prompt (>>>) instantly!

For Open WebUI users: If you use the browser UI, simply hit Refresh (F5) on your Open WebUI dashboard page after the terminal pull finishes. Your model dropdown is now automatically pointing to the updated file.